# Semantic Search with ChromaDB, a Vector Database


### Similar to Intro-to-Vector-DBs-with-ChromaDB.ipynb but with actual text files as input, found under 'inputText' folder.

---


---
### 1) Initialize the embedding model and Create Chroma Client

In [10]:
import os
from sentence_transformers import SentenceTransformer
import chromadb

# 1️⃣ Initialize model and Chroma
model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("text_files")


---
### 2) Read all .txt files from the folder

In [13]:
# 2️⃣ Read all .txt files from the folder
input_folder = f"C:\\Users\\yodah\\OneDrive\\Desktop\\Code-Work\\Python\\Practice\\Vector-DB-Practice\\inputText"
documents = []
file_ids = []

for filename in os.listdir(input_folder):
    if filename.endswith(".txt"):
        file_path = os.path.join(input_folder, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().strip()
            documents.append(content)
            file_ids.append(filename)  # Use filename as unique ID

print(f"Loaded {len(documents)} text files from '{input_folder}'")

Loaded 3 text files from 'C:\Users\yodah\OneDrive\Desktop\Code-Work\Python\Practice\Vector-DB-Practice\inputText'


---
### 3) Create embeddings



In [14]:
embeddings = model.encode(documents).tolist()


---

### 4) Store them in Chroma

#### Add the following:
- Documents (text content)
- Embeddings (vector representations of the text)
- IDs (unique identifiers for each document)

In [15]:
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=file_ids
)
print("All text files have been embedded and added to the vector database!")


All text files have been embedded and added to the vector database!


---

### 5) Perform a semantic search


In [18]:
query = "What is artificial intelligence?"
query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)


#### Search for similar documents given a query text

In [19]:
print("\n🔍 Search Results for query:", query)
for doc_id, doc_text in zip(results["ids"][0], results["documents"][0]):
    print(f"\n📄 File: {doc_id}\n{doc_text[:300]}...")  # show first 300 chars


🔍 Search Results for query: What is artificial intelligence?

📄 File: doc1.txt
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines. 
It involves processes such as learning, reasoning, and self-correction. 
Modern AI applications include natural language processing, image recognition, and autonomous vehicles. 
Machine learning, a subset of AI...

📄 File: doc3.txt
The stock market allows companies to raise capital and investors to earn returns. 
Market volatility often reflects changes in economic conditions, interest rates, and investor sentiment. 
Artificial intelligence and machine learning are increasingly used in algorithmic trading to identify patterns ...


---

### Explanation of Code



## 🧩 The Results Code Output

```python
print("\n🔍 Search Results for query:", query)
for doc_id, doc_text in zip(results["ids"][0], results["documents"][0]):
    print(f"\n📄 File: {doc_id}\n{doc_text[:300]}...")  # show first 300 chars
```

---

## 1️⃣ Where `results` Comes From

This line earlier in your code:

```python
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)
```

asks the vector database:

> “Find the 2 most similar documents to this query embedding.”

Chroma returns a **dictionary** that looks like this:

```python
{
  "ids": [["doc1.txt", "doc3.txt"]],
  "documents": [[
      "Artificial Intelligence (AI) refers to ...",
      "The stock market allows companies to raise capital ..."
  ]],
  "distances": [[0.12, 0.34]]
}
```

Each top-level list (`[ ... ]`) represents a *batch* of queries — since you could query multiple embeddings at once.

That’s why you see `[0]` everywhere:
`results["ids"][0]` → the IDs for the **first query**
`results["documents"][0]` → the matching texts for that query

---

## 2️⃣ The `zip()` Function

```python
for doc_id, doc_text in zip(results["ids"][0], results["documents"][0]):
```

Here’s what happens:

* `zip()` combines the two lists element-by-element:

  ```python
  zip(
    ["doc1.txt", "doc3.txt"],
    ["Artificial Intelligence ...", "The stock market allows ..."]
  )
  ```

  → becomes an iterable like:

  ```python
  [("doc1.txt", "Artificial Intelligence ..."),
   ("doc3.txt", "The stock market allows ...")]
  ```

So on each loop iteration:

* `doc_id` = `"doc1.txt"`
* `doc_text` = `"Artificial Intelligence (AI) refers to ..."`

---

## 3️⃣ The `print()` Formatting

```python
print(f"\n📄 File: {doc_id}\n{doc_text[:300]}...")
```

* `\n` → starts a new line (for nicer formatting)
* `📄 File: {doc_id}` → shows which file was matched
* `{doc_text[:300]}` → prints only the **first 300 characters** of the document to avoid flooding the console
* `...` → added manually to show there’s more text that’s not displayed

So you get neat, readable output:

```
📄 File: doc1.txt
Artificial Intelligence (AI) refers to the simulation ...
```

---

## 4️⃣ Why The Output Looks Like That

Your query was:

```
What is artificial intelligence?
```

The vector embedding of that sentence lives near texts that **talk about AI** in semantic space.
When Chroma compared your query embedding against the stored document embeddings, it found:

| Rank | File         | Similarity Reason                                                                           |
| ---- | ------------ | ------------------------------------------------------------------------------------------- |
| 🥇 1 | **doc1.txt** | It directly defines AI — perfect semantic match                                             |
| 🥈 2 | **doc3.txt** | It mentions “artificial intelligence” in the context of finance and trading, still relevant |
| 🥉 3 | **doc2.txt** | Talks about dogs, no semantic overlap, so excluded                                          |

That’s why your output looked like:

```
📄 File: doc1.txt
Artificial Intelligence (AI) refers to the simulation of human intelligence...

📄 File: doc3.txt
The stock market allows companies to raise capital... Artificial intelligence and machine learning...
```

---

## 🧠 Summary

| Code Part                         | Purpose                                                |
| --------------------------------- | ------------------------------------------------------ |
| `results = collection.query(...)` | Retrieves most similar docs                            |
| `results["ids"][0]`               | IDs of matching docs for this query                    |
| `results["documents"][0]`         | Text of those docs                                     |
| `zip()`                           | Pairs each ID with its text                            |
| `[:300]`                          | Truncates output for readability                       |
| Output order                      | Based on similarity to your query (closest → farthest) |

---

Would you like me to show you **how to print the similarity score (distance)** next to each file — so you can *see* how close each document was to the query numerically?
